In [1]:
import sys
sys.path.append('../..')

from src import utils
import src.channelCoding as cc
from pymongo import MongoClient
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import hmac


# MongoDB connection


client = MongoClient('mongodb://localhost:27017/')
# db = client['MAC_SC_R=1_2']
# TARGET_MAC = '3f8cd3da0bf84566839a7c790889884bab003e2de42c45d822188c14e1fbb719'
# MESSAGE = "This message is the default payload for the tests, and is 1088 bits long. It will be superposed with MAC tag of "


db = client['r0_r1_SC_alpha_0_3_R=0_5']
phase_SC_col = db['destination, phase_1']
phase_SC_col.find_one({}).keys()



/home/moh/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "
2025-08-05 12:26:13.514306: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-05 12:26:13.587502: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754414773.616540 1591110 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for

dict_keys(['_id', 'BER_tag', 'BER_msg', 'r0', 'r1', 'SNR', 'sigma_n2', 'config'])

In [2]:

df = pd.DataFrame(list(phase_SC_col.find({})))
df.head()

MAC_SIZE_ENCODED = int(np.ceil(256/df['config'][0]["MAC_REP"]/df['config'][0]["MAC_LDPC"]))
Payload = df['config'][0]['PAYLOAD'][:(MAC_SIZE_ENCODED - 512) //16]
m = utils.string_to_bits(Payload)
msg = cc.encode_LDPC(m,MAC_SIZE_ENCODED)
msg = np.array(msg)

tag_hex = hmac.new(df['config'][0]['MAC_KEY'].encode(), Payload.encode(), 'sha256').hexdigest()
n = utils.hex_to_bits(tag_hex)
tag_LDPC = cc.encode_LDPC(n, int(np.round(len(n)/df['config'][0]['MAC_LDPC'])))
tag_REP = np.repeat(tag_LDPC, int(np.round(1/df['config'][0]['MAC_REP'])))
tag = np.array(tag_REP)



I0000 00:00:1754414804.673244 1591110 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 20925 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:01:00.0, compute capability: 8.6


In [ ]:
df.head()

,_id,BER_tag,BER_msg,r0,r1,SNR,sigma_n2,config
0,688a99b5e0a4e198d772a65a,0.000000,0.000000,"[0.14299277743298047, 0.021179754467207122, 0....","[8.009067317466152e-05, 0.04908598982432574, 0...",19.768045,1.099435e-07,"{'_id': 687ebcefbe94ff982757e758, 'MongoDB_Col..."
1,688a99b9e0a4e198d772a65c,0.000000,0.000000,"[0.17022096546535853, 0.024702646320288336, 0....","[0.0005212420167197899, 0.04637638932884151, 0...",19.745886,1.099434e-07,"{'_id': 687ebcefbe94ff982757e758, 'MongoDB_Col..."
2,688a99c8e0a4e198d772a660,0.000137,0.000137,"[0.1008404538862248, 0.013029857306028559, 0.0...","[6.195329824773142e-05, 0.03964589086560142, 0...",18.483547,1.079805e-07,"{'_id': 687ebcefbe94ff982757e758, 'MongoDB_Col..."
3,688a99cce0a4e198d772a662,0.000137,0.000000,"[0.1264763771014677, 0.016484577806900436, 0.0...","[0.00035394083961520014, 0.047363655838376324,...",19.521638,1.079571e-07,"{'_id': 687ebcefbe94ff982757e758, 'MongoDB_Col..."
4,688a99f1e0a4e198d772a66a,0.000000,0.000000,"[0.14077378929945236, 0.014102847400439319, 0....","[0.00046899829790621585, 0.05302016004433801, ...",19.485498,1.096595e-07,"{'_id': 687ebcefbe94ff982757e758, 'MongoDB_Col..."


: 

In [ ]:

# # — 1) first, pull α out into its own column (adjust this to how α really lives in your config) —
# df['alpha'] = df['config'].apply(lambda c: c['ALPHA'])  

# rows = []
# for idx, row in df.iterrows():
#     E0_list = row['r0']
#     E1_list = row['r1']

#     alpha    = row['alpha']
#     snr  = row['SNR']
#     sigma_n_2   = row['sigma_n2']
    
#     # Sanity check:
#     assert len(E0_list) == len(E1_list) == len(msg) == len(tag)
    
#     for e0, e1, m, t in zip(E0_list, E1_list, msg, tag):
#         rows.append({
#             'E0'       : e0,
#             'E1'       : e1,
#             'alpha'    : alpha,
#             'SNR'      : snr,
#             'sigma_n2' : sigma_n_2,
#             'msg_label': m,
#             'tag_label': t
#         })

# # build the new “bit-level” DataFrame
# bit_df = pd.DataFrame(rows)



from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, arrays_zip, col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression

# 1) bootstrap Spark
spark = (
    SparkSession.builder
      .appName("FSK_Superposition_Bit_Dataset")
      # if reading from MongoDB, add the mongo-spark connector configs:
      # .config("spark.mongodb.input.uri", "mongodb://127.0.0.1/mydb.mycoll")
      .getOrCreate()
)

# 2) load your packet DataFrame
# Option A: from Parquet
df = spark.read.parquet("s3://…/fsk_packets.parquet")

# Option B: directly from MongoDB (requires mongo-spark connector on your CLASSPATH)
# df = spark.read.format("mongo").load()

# keep only the columns we need
df = df.select("r0", "r1", "msg_bits", "tag_bits", "alpha", "SNR", "sigma_n2")

# 3) zip the four arrays into one array-of-structs, then explode
df_bits = (
    df
    .withColumn("zipped",
        arrays_zip("r0","r1","msg_bits","tag_bits")
    )
    .withColumn("bit", explode("zipped"))
    .select(
        col("bit.r0"      ).alias("E0"),
        col("bit.r1"      ).alias("E1"),
        col("bit.msg_bits").alias("msg_label"),
        col("bit.tag_bits").alias("tag_label"),
        "alpha", "SNR", "sigma_n2"
    )
)

# 4) (optional) sample, e.g. keep 10% of all bits at random
df_sampled = df_bits.sample(withReplacement=False, fraction=0.1, seed=42)

# 5) assemble your features for MLlib
assembler = VectorAssembler(
    inputCols=["E0","E1","alpha","SNR","sigma_n2"],
    outputCol="features"
)
df_feat = assembler.transform(df_sampled).select("features","msg_label","tag_label")

# 6) example: train two logistic‐regression models, one for msg, one for tag
lr_msg = LogisticRegression(labelCol="msg_label", featuresCol="features")
lr_tag = LogisticRegression(labelCol="tag_label", featuresCol="features")

msg_model = lr_msg.fit(df_feat)
tag_model = lr_tag.fit(df_feat)

# now msg_model and tag_model are fitted—you can evaluate or save them

